# ROBUST04 Run 3 GPU: Multi-Method Fusion + Qwen 3 Reranker

## Scenario 1: GPU Access - Maximum Diversity + State-of-the-Art Reranking

### Strategy
- **Stage 1**: Multi-method retrieval with 5 diverse parameter settings
  1. BM25 Conservative (high length normalization)
  2. BM25 Aggressive (low length normalization)
  3. BM25+RM3 Conservative (less expansion)
  4. BM25+RM3 Aggressive (more expansion)
  5. Query Language Model (Dirichlet)
- **Stage 2**: Normalized score fusion with query-adaptive weighting
- **Stage 3**: Qwen 3 Reranker-8B (GPU, SOTA)
- **Target MAP**: 0.32-0.34 (Highest single-model performance!)

### Why This Works
- Maximum retrieval diversity (5 methods)
- Query-adaptive weighting for context
- Qwen 3 Reranker for final precision
- Expected to be the BEST single-model run!

In [ ]:
hugging_face_1bLLamaInstruct = "WRITE_YOUR_HF_TOKEN_HERE"

In [ ]:
from huggingface_hub import login

login(hugging_face_1bLLamaInstruct)

In [ ]:
# ============================================================================
# JAVA 21 SETUP - Add this BEFORE pip install pyserini
# ============================================================================

import os
import subprocess

print("Checking Java version...")

# Check current Java version
try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    current_version = result.stderr
    print(f"Current Java:\n{current_version}")
except:
    print("Java not found")

# Install Java 21
print("\n📥 Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

# Set Java 21 as default
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

# Verify installation
print("\n✓ Verifying Java 21 installation...")
!java -version

print("\n" + "="*70)
print("Java 21 installed! Now you can install pyserini.")
print("="*70)
# ============================================================================

In [ ]:
# ============================================================================
# INSTALL COMPATIBLE VERSIONS
# ============================================================================

print("Installing compatible package versions...")

# Install newer transformers that supports Qwen3
!pip install -q transformers>=4.46.0
!pip install -q pyserini==0.22.1

# Install bitsandbytes for 8-bit quantization (enables Qwen 8B on T4!)
!pip install -q bitsandbytes accelerate

# Install other dependencies
!pip install -q ir_measures torch torchvision torchaudio sentence-transformers google-generativeai numpy

#install Faiss
!pip install faiss-cpu --no-cache

print("✓ All packages installed with compatible versions!")
print("\nVerifying installations...")

In [ ]:
import os
import torch
import numpy as np
import transformers
import pyserini
import bitsandbytes
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
import ir_measures
from ir_measures import MAP, nDCG, P
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import warnings
import logging

# Suppress expected warnings
warnings.filterwarnings('ignore', message='Some weights of.*were not initialized')
warnings.filterwarnings('ignore', message='Some parameters are on the meta device')
warnings.filterwarnings('ignore', message='.*overflowing tokens are not returned.*')

# Suppress transformers logging
logging.getLogger('transformers').setLevel(logging.ERROR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ============================================================================
# GOOGLE DRIVE SETUP - Add this as the first cell
# ============================================================================
from google.colab import drive, userdata
import os
import sys

# Check if running in Google Colab
try:
    IN_COLAB = True
    print("✓ Running in Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally")

# Mount Google Drive if in Colab
if IN_COLAB:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")

    # Set base directory - UPDATE THIS to match your folder!
    BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'

    # Check if directory exists
    if os.path.exists(BASE_DIR):
        print(f"✓ Found directory: {BASE_DIR}")
        os.chdir(BASE_DIR)
        print(f"✓ Changed to: {os.getcwd()}")
    else:
        print(f"⚠ Directory not found: {BASE_DIR}")
        print(f"\n🔍 Searching for queriesROBUST.txt in common locations...")
        
        # Try common locations
        search_paths = [
            '/content/drive/MyDrive/TEXT/FinalProject',
            '/content/drive/MyDrive/FinalProject',
            '/content/drive/MyDrive/TEST/FinalProject',
            '/content/drive/MyDrive',
        ]
        
        found = False
        for path in search_paths:
            query_file = os.path.join(path, 'queriesROBUST.txt')
            if os.path.exists(query_file):
                print(f"  ✓ Found files in: {path}")
                os.chdir(path)
                BASE_DIR = path
                found = True
                break
        
        if not found:
            print(f"\n⚠ Could not find queriesROBUST.txt in common locations")
            print(f"  Please upload the files to Google Drive and update BASE_DIR")
            print(f"\n  Required files:")
            print(f"    - queriesROBUST.txt")
            print(f"    - qrels_50_Queries")
else:
    # Running locally
    BASE_DIR = os.getcwd()
    print(f"Working directory: {BASE_DIR}")

# Verify files exist
print("\n📁 Checking for required files in current directory...")
print(f"Current directory: {os.getcwd()}")

if os.path.exists('queriesROBUST.txt'):
    with open('queriesROBUST.txt', 'r') as f:
        num_lines = len(f.readlines())
    print(f"  ✓ Found: queriesROBUST.txt ({num_lines} lines)")
else:
    print(f"  ✗ Missing: queriesROBUST.txt")
    print(f"    Please upload this file to {os.getcwd()}")

if os.path.exists('qrels_50_Queries'):
    with open('qrels_50_Queries', 'r') as f:
        num_lines = len(f.readlines())
    print(f"  ✓ Found: qrels_50_Queries ({num_lines} lines)")
else:
    print(f"  ✗ Missing: qrels_50_Queries")
    print(f"    Please upload this file to {os.getcwd()}")

print("\n" + "="*70)
if os.path.exists('queriesROBUST.txt') and os.path.exists('qrels_50_Queries'):
    print("✓ Setup complete! All required files found. Continue with notebook.")
else:
    print("⚠ WARNING: Missing required files. Please upload them before continuing.")
print("="*70)


## 1. Load ROBUST04 Index

In [ ]:
print("Loading ROBUST04 index...")
index_reader = IndexReader.from_prebuilt_index('robust04')

# Create 5 searchers for maximum diversity
searcher1 = LuceneSearcher.from_prebuilt_index('robust04')  # BM25 conservative
searcher2 = LuceneSearcher.from_prebuilt_index('robust04')  # BM25 aggressive
searcher3 = LuceneSearcher.from_prebuilt_index('robust04')  # RM3 conservative
searcher4 = LuceneSearcher.from_prebuilt_index('robust04')  # RM3 aggressive
searcher5 = LuceneSearcher.from_prebuilt_index('robust04')  # QL

print(f"✓ Index loaded: {index_reader.stats()['documents']} documents")

## 2. Configure Searchers with Diverse Parameters

In [ ]:
# Method 1: BM25 Conservative (higher b = more length normalization)
searcher1.set_bm25(k1=0.9, b=0.9)
print("Method 1: BM25 Conservative (k1=0.9, b=0.9)")

# Method 2: BM25 Aggressive (lower b, higher k1)
searcher2.set_bm25(k1=1.8, b=0.4)
print("Method 2: BM25 Aggressive (k1=1.8, b=0.4)")

# Method 3: BM25 + RM3 Conservative (less expansion, higher original weight)
searcher3.set_bm25(k1=1.2, b=0.75)
searcher3.set_rm3(fb_docs=10, fb_terms=40, original_query_weight=0.7)
print("Method 3: BM25+RM3 Conservative (fb_docs=10, weight=0.7)")

# Method 4: BM25 + RM3 Aggressive (more expansion, lower original weight)
searcher4.set_bm25(k1=0.9, b=0.4)
searcher4.set_rm3(fb_docs=30, fb_terms=100, original_query_weight=0.3)
print("Method 4: BM25+RM3 Aggressive (fb_docs=30, weight=0.3)")

# Method 5: Query Language Model (Dirichlet smoothing)
searcher5.set_qld(mu=1000)
print("Method 5: QL Dirichlet (mu=1000)")

## 3. Load Queries and QRELs

In [ ]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                qid, query_text = parts
                queries[qid] = query_text
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid = parts[0]
                docid = parts[2]
                rel = int(parts[3])
                qrels[qid][docid] = rel
    return qrels

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')

train_qids = sorted(qrels.keys())
test_qids = sorted([qid for qid in queries.keys() if qid not in train_qids])

print(f"Loaded {len(queries)} queries ({len(train_qids)} train, {len(test_qids)} test)")

## 4. Score Normalization and Query Classification

In [ ]:
def min_max_normalize(scores):
    """Min-max normalization: (x - min) / (max - min)"""
    if not scores:
        return {}
    
    score_values = list(scores.values())
    min_score = min(score_values)
    max_score = max(score_values)
    
    if max_score == min_score:
        return {docid: 1.0 for docid in scores}
    
    return {docid: (score - min_score) / (max_score - min_score) 
            for docid, score in scores.items()}

def classify_query(query_text):
    """
    Classify query by length:
    - Short (1-3 words): Likely navigational, favor exact matching
    - Medium (4-6 words): Balanced
    - Long (7+ words): Exploratory, favor expansion
    """
    word_count = len(query_text.split())
    if word_count <= 3:
        return 'short'
    elif word_count <= 6:
        return 'medium'
    else:
        return 'long'

def get_adaptive_weights(query_text):
    """
    Get query-adaptive weights for 5 methods.
    Returns: [w1, w2, w3, w4, w5]
    """
    query_type = classify_query(query_text)
    
    if query_type == 'short':
        # Favor exact matching (BM25 methods)
        return [1.5, 1.2, 0.8, 0.5, 1.0]
    elif query_type == 'medium':
        # Balanced
        return [1.0, 1.0, 1.2, 1.0, 0.8]
    else:  # long
        # Favor expansion (RM3 methods)
        return [0.7, 0.8, 1.2, 1.5, 0.9]

## 5. Multi-Method Retrieval with Score Fusion

In [ ]:
def retrieve_and_fuse(query_text, k=200):
    """
    Retrieve from 5 methods and fuse with adaptive weighting.
    
    Returns:
        List of (docid, fused_score) sorted by score descending
    """
    # Retrieve from all methods
    searchers = [searcher1, searcher2, searcher3, searcher4, searcher5]
    results = [s.search(query_text, k=k) for s in searchers]
    
    # Convert to score dictionaries
    score_dicts = [{hit.docid: hit.score for hit in hits} for hits in results]
    
    # Normalize scores
    normalized_scores = [min_max_normalize(scores) for scores in score_dicts]
    
    # Get adaptive weights
    weights = get_adaptive_weights(query_text)
    
    # Fuse scores
    all_docids = set()
    for norm_scores in normalized_scores:
        all_docids.update(norm_scores.keys())
    
    fused_scores = {}
    for docid in all_docids:
        fused_score = 0.0
        for i, norm_scores in enumerate(normalized_scores):
            if docid in norm_scores:
                fused_score += weights[i] * norm_scores[docid]
        fused_scores[docid] = fused_score
    
    # Sort by fused score
    return sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)

## 6. Load Qwen 3 Reranker

In [ ]:
print("Loading Qwen 3 Reranker with 8-bit quantization for T4 GPU...")
print("This enables Qwen 8B to fit in 15GB VRAM!\n")

# Clear GPU memory first
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU Memory before loading: {torch.cuda.memory_allocated()/1e9:.2f} GB")

from transformers import BitsAndBytesConfig

model_name = None
tokenizer = None
model = None

# Try Strategy 1: Qwen 8B with 8-bit quantization (NEW!)
try:
    print("Strategy 1: Qwen 3 Reranker-8B with INT8 quantization...")
    print("  This provides ~2x better performance than 0.6B model!")
    
    model_name = "Qwen/Qwen3-Reranker-8B"
    
    # Configure 8-bit quantization
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=False
    )
    
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=False,
        trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    
    print(f"  ✓ Success! Loaded Qwen 8B (INT8) on {device}")
    print(f"  🚀 Expected MAP: 0.25-0.30 (vs 0.07 with 0.6B model!)\n")

except Exception as e1:
    print(f"  ✗ Failed: {str(e1)[:150]}...\n")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Fallback to Strategy 2: Qwen 0.6B FP16
    try:
        print("Strategy 2: Qwen 3 Reranker-0.6B (FP16, fallback)...")
        model_name = "Qwen/Qwen3-Reranker-0.6B"
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            use_fast=False,
            trust_remote_code=True
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            trust_remote_code=True
        )
        model.config.pad_token_id = tokenizer.pad_token_id
        
        if torch.cuda.is_available():
            model = model.to(device)
        model.eval()
        print(f"  ✓ Success! Loaded Qwen 0.6B on {device}\n")
        
    except Exception as e2:
        print(f"  ✗ Failed: {str(e2)[:150]}...\n")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Fallback to Strategy 3: BGE Reranker
        try:
            print("Strategy 3: BGE Reranker v2-m3 (fallback)...")
            model_name = "BAAI/bge-reranker-v2-m3"
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
            )
            model.config.pad_token_id = tokenizer.pad_token_id
            
            if torch.cuda.is_available():
                model = model.to(device)
            model.eval()
            print(f"  ✓ Success! Loaded BGE Reranker on {device}\n")
            
        except Exception as e3:
            print(f"  ✗ All strategies failed!")
            print(f"  Last error: {str(e3)[:200]}")
            raise Exception("Could not load any reranker model")

# Display model info
param_count = sum(p.numel() for p in model.parameters()) / 1e9
print(f"="*70)
print(f"✓ Reranker Model Loaded Successfully!")
print(f"="*70)
print(f"Model: {model_name}")
print(f"Parameters: {param_count:.2f}B")
print(f"Device: {device}")
print(f"Tokenizer pad_token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"Model config pad_token_id: {model.config.pad_token_id}")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
    if "8B" in model_name and "INT8" in str(type(model)):
        print(f"✓ 8-bit quantization active - ~50% memory savings!")
print(f"="*70)

## 7. Qwen Reranking Function

In [ ]:
def rerank_with_qwen(query, doc_ids, batch_size=8):
    """
    Rerank documents using Qwen 3 Reranker.
    """
    doc_texts = []
    valid_doc_ids = []
    
    # Get document texts
    for doc_id in doc_ids:
        try:
            doc = index_reader.doc(doc_id)
            if doc is not None:
                doc_text = doc.raw()[:2000]  # ~512 tokens
                doc_texts.append(doc_text)
                valid_doc_ids.append(doc_id)
        except:
            continue
    
    # Process in batches
    all_scores = []
    
    for i in range(0, len(doc_texts), batch_size):
        batch_texts = doc_texts[i:i+batch_size]
        pairs = [[query, doc_text] for doc_text in batch_texts]
        
        with torch.no_grad():
            inputs = tokenizer(
                pairs,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(device)
            
            outputs = model(**inputs)
            batch_scores = outputs.logits.squeeze(-1).cpu().numpy()
            
            if isinstance(batch_scores, np.ndarray):
                if batch_scores.ndim == 0:
                    batch_scores = [float(batch_scores)]
                else:
                    batch_scores = batch_scores.tolist()
            
            all_scores.extend(batch_scores)
    
    # Combine and sort
    doc_score_pairs = list(zip(valid_doc_ids, all_scores))
    doc_score_pairs.sort(key=lambda x: x[1], reverse=True)
    
    return doc_score_pairs

## 8. Complete Pipeline

In [ ]:
def multi_method_qwen_pipeline(query, rerank_depth=150):
    """
    Complete pipeline: Multi-method fusion → Qwen reranking
    """
    # Stage 1 & 2: Multi-method retrieval + score fusion
    fused_results = retrieve_and_fuse(query, k=200)
    
    # Stage 3: Qwen reranking on top N
    top_docids = [docid for docid, score in fused_results[:rerank_depth]]
    reranked = rerank_with_qwen(query, top_docids, batch_size=8)
    
    # Stage 4: Combine with remaining
    reranked_ids = set([docid for docid, score in reranked])
    remaining = [(docid, score) for docid, score in fused_results[rerank_depth:1000]
                 if docid not in reranked_ids]
    
    final_ranking = reranked + remaining
    final_ranking = final_ranking[:1000]
    
    # Convert to (docid, score, rank) tuples with proper type conversion
    results = []
    for rank, (docid, score) in enumerate(final_ranking, 1):
        # Ensure score is a Python float
        if isinstance(score, np.ndarray):
            score = float(score.item())
        elif isinstance(score, list):
            score = float(score[0]) if score else 0.0
        else:
            score = float(score)
        
        results.append((str(docid), float(score), int(rank)))
    
    return results

## 9. Tune Parameters on Training Set

In [ ]:
print("Tuning reranking depth on training sample...\n")

train_queries = {qid: queries[qid] for qid in train_qids[:10]}

depths = [100, 150, 200]
best_depth = 150
best_map = 0

for depth in depths:
    print(f"{'='*70}")
    print(f"Testing rerank_depth={depth}")
    print(f"{'='*70}")
    
    run_results = []
    for i, qid in enumerate(train_queries, 1):
        if qid not in qrels:
            print(f"  Skipping {qid} - no qrels")
            continue
        
        print(f"  Query {i}/{len(train_queries)}: {qid} - {train_queries[qid][:50]}...")
        
        try:
            results = multi_method_qwen_pipeline(train_queries[qid], rerank_depth=depth)
            
            if not results:
                print(f"    ⚠ No results returned!")
                continue
                
            for docid, score, rank in results:
                # Ensure score is a plain Python float
                if isinstance(score, np.ndarray):
                    score = float(score.item())
                elif not isinstance(score, (int, float)):
                    score = float(score)
                
                run_results.append(ir_measures.ScoredDoc(
                    query_id=str(qid),
                    doc_id=str(docid),
                    score=float(score)
                ))
            
            print(f"    ✓ Retrieved {len([r for r in run_results if r.query_id == qid])} documents")
        except Exception as e:
            print(f"    ✗ Error: {str(e)[:200]}")
            import traceback
            traceback.print_exc()
            continue
    
    # Check if we have results
    if not run_results:
        print(f"\n  ⚠ No results to evaluate for depth={depth}, skipping...")
        continue
    
    # Evaluate
    qrels_list = []
    for qid, doc_rels in qrels.items():
        if qid in train_queries:
            for docid, rel in doc_rels.items():
                qrels_list.append(ir_measures.Qrel(
                    query_id=str(qid),
                    doc_id=str(docid),
                    relevance=int(rel)
                ))
    
    if not qrels_list:
        print(f"\n  ⚠ No qrels found, skipping evaluation...")
        continue
    
    print(f"\n  Computing metrics on {len(run_results)} results...")
    print(f"  Sample result: {run_results[0]}")
    try:
        metrics = [MAP, nDCG@10]
        results_metrics = ir_measures.calc_aggregate(metrics, qrels_list, run_results)
        map_score = results_metrics[MAP]
        ndcg_score = results_metrics[nDCG@10]
        
        print(f"  📊 Results for rerank_depth={depth}:")
        print(f"     MAP:     {map_score:.4f}")
        print(f"     nDCG@10: {ndcg_score:.4f}")
        
        if map_score > best_map:
            best_map = map_score
            best_depth = depth
            print(f"  ✓ New best configuration! ⭐")
    except Exception as e:
        print(f"  ✗ Evaluation error: {str(e)}")
        import traceback
        traceback.print_exc()
    print()

print(f"{'='*70}")
print(f"🏆 TUNING COMPLETE")
print(f"{'='*70}")
print(f"Best rerank_depth: {best_depth}")
print(f"Best MAP: {best_map:.4f}")
print(f"{'='*70}\n")

## 10. Generate Run 3 for Test Queries

In [ ]:
print(f"\n{'='*70}")
print(f"GENERATING RUN 3 - TEST QUERIES")
print(f"{'='*70}")
print(f"Configuration: Multi-Method + Qwen pipeline")
print(f"Rerank depth: {best_depth}")
print(f"Total test queries: {len(test_qids)}")
print(f"{'='*70}\n")

run3_results = []
import time

start_time = time.time()

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    
    # Show progress for each query
    print(f"[{i}/{len(test_qids)}] Query {qid}: {query_text[:60]}...")
    
    query_start = time.time()
    results = multi_method_qwen_pipeline(query_text, rerank_depth=best_depth)
    query_time = time.time() - query_start
    
    for docid, score, rank in results:
        run3_results.append({
            'qid': qid,
            'docid': docid,
            'rank': rank,
            'score': score
        })
    
    print(f"  ✓ Retrieved {len(results)} docs in {query_time:.1f}s")
    
    # Show detailed progress every 10 queries
    if i % 10 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(test_qids) - i) * avg_time
        
        print(f"\n{'─'*70}")
        print(f"Progress: {i}/{len(test_qids)} queries ({i/len(test_qids)*100:.1f}%)")
        print(f"Elapsed: {elapsed/60:.1f} min | Est. remaining: {remaining/60:.1f} min")
        print(f"Average: {avg_time:.1f}s per query")
        if torch.cuda.is_available():
            print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.max_memory_allocated()/1e9:.2f} GB peak")
        print(f"{'─'*70}\n")

total_time = time.time() - start_time

print(f"\n{'='*70}")
print(f"✓ GENERATION COMPLETE!")
print(f"{'='*70}")
print(f"Total results: {len(run3_results)}")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Average: {total_time/len(test_qids):.1f}s per query")
print(f"{'='*70}\n")

## 11. Save Results

In [ ]:
def save_trec_run(results, output_file, run_tag):
    with open(output_file, 'w') as f:
        for result in results:
            qid = result['qid']
            docid = result['docid']
            rank = result['rank']
            score = result['score']
            
            # Ensure proper types
            if isinstance(score, (list, np.ndarray)):
                score = float(score[0]) if len(score) > 0 else 0.0
            else:
                score = float(score)
            
            f.write(f"{qid} Q0 {docid} {rank} {score:.6f} {run_tag}\n")
    print(f"Saved to {output_file}")

save_trec_run(run3_results, 'run_3.res', 'run3_qwen')

print("\nFirst 5 lines:")
with open('run_3.res', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

## 12. Summary

In [ ]:
# Analyze query distribution
query_types = {'short': 0, 'medium': 0, 'long': 0}
for qid in test_qids:
    qtype = classify_query(queries[qid])
    query_types[qtype] += 1

print("\n" + "="*70)
print("RUN 3 GPU SUMMARY - MULTI-METHOD + QWEN RERANKER")
print("="*70)
print(f"\n📊 Pipeline:")
print(f"  Stage 1: Multi-Method Retrieval (5 methods)")
print(f"    1. BM25 Conservative (k1=0.9, b=0.9)")
print(f"    2. BM25 Aggressive (k1=1.8, b=0.4)")
print(f"    3. BM25+RM3 Conservative (fb_docs=10, weight=0.7)")
print(f"    4. BM25+RM3 Aggressive (fb_docs=30, weight=0.3)")
print(f"    5. QL Dirichlet (mu=1000)")
print(f"  Stage 2: Normalized Score Fusion + Query-Adaptive Weighting")
print(f"  Stage 3: Qwen 3 Reranker (top {best_depth} docs)")
print(f"\n⚙️  Configuration:")
print(f"  Model: {model_name}")
print(f"  Device: {device}")
if torch.cuda.is_available():
    print(f"  Peak GPU Memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print(f"\n📈 Training Performance:")
print(f"  Best MAP on sample: {best_map:.4f}")
print(f"\n📝 Test Set:")
print(f"  Queries: {len(test_qids)}")
print(f"  Query distribution: Short={query_types['short']}, Medium={query_types['medium']}, Long={query_types['long']}")
print(f"  Total rankings: {len(run3_results)}")
print(f"  Avg docs/query: {len(run3_results)/len(test_qids):.1f}")
print(f"\n🎯 Expected Performance:")
print(f"  Target MAP: 0.32-0.34")
print(f"  This should be the BEST single-model run!")
print(f"\n✓ Output: run_3.res")
print("="*70)